### Timeouts

In [ ]:
import asyncio
import random
import time

start_time = time.monotonic()

def log(message: str) -> None:
    elapsed = time.monotonic() - start_time
    print(f"[{elapsed:6.2f}s] {message}")

In [ ]:
async def make_item(item: str = "coffee", brew: float = 2.0, fail: bool = False) -> str:
    try:
        await asyncio.sleep(brew)
    except asyncio.CancelledError:
        log(f"{item}: thrown out")
        raise
    if fail:
        raise RuntimeError(f"out of {item}")
    return item

**`wait_for`**

In [ ]:
start_time = time.monotonic()

brew = random.choice([0.5, 1.0, 5.0])
log(f"Beanery: will take {brew}s")
try:
    coffee = await asyncio.wait_for(make_item("Beanery", brew=brew), timeout=2.0)
    log(f"got: {coffee}")
except asyncio.TimeoutError:
    log("too slow, making instant coffee")

**`async with asyncio.timeout(...)` — preferred modern style**

In [ ]:
start_time = time.monotonic()

brew = random.choice([0.5, 1.0, 5.0])
log(f"Roastery: will take {brew}s")
try:
    async with asyncio.timeout(2.0):
        coffee = await make_item("Roastery", brew=brew)
        log(f"got: {coffee}")
except asyncio.TimeoutError:
    log("deadline missed")

**Retry**

In [ ]:
async def brew_with_retry(cafe: str, attempts: int = 4, per_attempt_timeout: float = 2.0) -> str:
    for attempt in range(1, attempts + 1):
        brew = random.choice([0.5, 1.0, 5.0])
        log(f"attempt {attempt}/{attempts} at {cafe}: will take {brew}s")
        try:
            async with asyncio.timeout(per_attempt_timeout):
                return await make_item(cafe, brew=brew)
        except asyncio.TimeoutError:
            log(f"attempt {attempt}: too slow")
    raise RuntimeError(f"{cafe}: gave up")


start_time = time.monotonic()
try:
    coffee = await brew_with_retry("Grindhouse")
    log(f"success: {coffee}")
except RuntimeError as error:
    log(f"failure: {error}")

---
- `wait_for` and `async with asyncio.timeout(...)` cancel and raise `TimeoutError` on deadline.
- Combine `timeout` with a retry loop for flaky services.